# Module 7: Inference Engine

This notebook covers efficient inference with nanochat's engine, including KV caching and tool use.

## What You'll Learn

- **Autoregressive generation** - Token-by-token prediction
- **KV Cache** - Avoiding redundant computation
- **Prefill vs Decode** - Two phases of generation
- **Sampling strategies** - Temperature, top-k
- **Tool use** - Calculator and Python execution

This module connects to **mini-sglang** concepts for advanced inference optimization.

In [ ]:
import torch
import torch.nn.functional as F
import math
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 7.1 Autoregressive Generation

LLMs generate text one token at a time, using previous tokens to predict the next:

```
Input:  [The, quick, brown]
Output: [fox]  ← predicted next token

Input:  [The, quick, brown, fox]
Output: [jumps]  ← predicted next token
...
```

In [ ]:
# Simple autoregressive generation (naive, no caching)
def naive_generate(model, input_ids, max_new_tokens, temperature=1.0):
    """
    Generate tokens one at a time.
    This is SLOW because we recompute everything each step.
    """
    generated = input_ids.clone()
    
    for _ in range(max_new_tokens):
        # Forward pass on ALL tokens
        logits = model(generated)  # (B, T, vocab_size)
        
        # Get logits for last position only
        next_logits = logits[:, -1, :]  # (B, vocab_size)
        
        # Sample next token
        if temperature > 0:
            probs = F.softmax(next_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        
        # Append to sequence
        generated = torch.cat([generated, next_token], dim=1)
    
    return generated

print("Problem with naive approach:")
print("  - Step 1: Compute attention for positions [0, 1, ..., n]")
print("  - Step 2: Compute attention for positions [0, 1, ..., n, n+1]")
print("  - Step 3: Compute attention for positions [0, 1, ..., n, n+1, n+2]")
print("  - ...")
print("\nWe keep recomputing the same attention for early positions!")
print("\nComplexity: O(n²) per step, O(n³) total")

## 7.2 KV Cache: The Key Optimization

The KV cache stores computed Key and Value tensors, so we only need to:
1. Compute Q, K, V for the **new** token
2. Append K, V to cache
3. Attend new Q to **all** cached K, V

In [ ]:
class KVCache:
    """
    Key-Value cache for efficient autoregressive generation.
    Stores K and V tensors for each layer.
    """
    
    def __init__(self, batch_size, num_heads, max_seq_len, head_dim, num_layers):
        self.shape = (num_layers, 2, batch_size, num_heads, max_seq_len, head_dim)
        self.cache = None  # Lazily initialized
        self.pos = 0  # Current position in cache
    
    def reset(self):
        self.pos = 0
    
    def get_pos(self):
        return self.pos
    
    def insert_kv(self, layer_idx, k, v):
        """
        Insert new K, V into cache and return full cached K, V.
        k, v: (batch, num_heads, seq_len_new, head_dim)
        """
        # Lazy init
        if self.cache is None:
            self.cache = torch.zeros(self.shape, dtype=k.dtype, device=k.device)
        
        # Get dimensions
        B, H, T_new, D = k.size()
        t0, t1 = self.pos, self.pos + T_new
        
        # Insert into cache
        self.cache[layer_idx, 0, :, :, t0:t1, :] = k  # Keys
        self.cache[layer_idx, 1, :, :, t0:t1, :] = v  # Values
        
        # Update position after last layer
        if layer_idx == self.shape[0] - 1:
            self.pos = t1
        
        # Return full cached K, V
        return self.cache[layer_idx, 0, :, :, :t1, :], self.cache[layer_idx, 1, :, :, :t1, :]

# Demonstrate cache usage
cache = KVCache(
    batch_size=1,
    num_heads=8,
    max_seq_len=100,
    head_dim=64,
    num_layers=4
)

# Simulate prefill (10 tokens)
k_prefill = torch.randn(1, 8, 10, 64)
v_prefill = torch.randn(1, 8, 10, 64)

for layer in range(4):
    k_cached, v_cached = cache.insert_kv(layer, k_prefill, v_prefill)

print(f"After prefill:")
print(f"  Cache position: {cache.get_pos()}")
print(f"  Cached K shape: {k_cached.shape}")

# Simulate decode (1 token at a time)
for step in range(3):
    k_new = torch.randn(1, 8, 1, 64)
    v_new = torch.randn(1, 8, 1, 64)
    
    for layer in range(4):
        k_cached, v_cached = cache.insert_kv(layer, k_new, v_new)
    
    print(f"After decode step {step+1}: pos={cache.get_pos()}, K shape={k_cached.shape}")

## 7.3 Prefill vs Decode Phases

Generation has two distinct phases:

1. **Prefill**: Process all input tokens in parallel
2. **Decode**: Generate output tokens one at a time

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize compute patterns
def visualize_phases():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Prefill: process all input tokens
    ax = axes[0]
    input_len = 10
    mask = np.tril(np.ones((input_len, input_len)))
    ax.imshow(mask, cmap='Blues')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    ax.set_title(f'Prefill Phase\n(Process {input_len} tokens in parallel)')
    for i in range(input_len):
        for j in range(input_len):
            if mask[i, j] > 0:
                ax.text(j, i, '✓', ha='center', va='center', fontsize=8)
    
    # Decode: one token at a time
    ax = axes[1]
    decode_steps = 5
    total_len = input_len + decode_steps
    decode_mask = np.zeros((decode_steps, total_len))
    for i in range(decode_steps):
        decode_mask[i, :input_len + i + 1] = 1
    ax.imshow(decode_mask, cmap='Oranges', aspect='auto')
    ax.set_xlabel('Key position (including cached)')
    ax.set_ylabel('Decode step')
    ax.set_title(f'Decode Phase\n(Generate {decode_steps} tokens, one at a time)')
    ax.axvline(x=input_len - 0.5, color='red', linestyle='--', label='Prefill boundary')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

visualize_phases()

print("Prefill phase:")
print("  - Compute-bound (large matrix operations)")
print("  - All input tokens processed in parallel")
print("  - KV cache is populated")
print("\nDecode phase:")
print("  - Memory-bound (reading KV cache)")
print("  - One token generated at a time")
print("  - KV cache grows by 1 each step")

## 7.4 Sampling Strategies

How we select the next token from logits:

In [ ]:
def sample_next_token(logits, temperature=1.0, top_k=None):
    """
    Sample next token from logits.
    
    Args:
        logits: (batch, vocab_size)
        temperature: Higher = more random, Lower = more deterministic
        top_k: Only consider top-k tokens
    """
    # Temperature = 0: greedy (always pick most likely)
    if temperature == 0:
        return torch.argmax(logits, dim=-1, keepdim=True)
    
    # Apply top-k filtering
    if top_k is not None:
        k = min(top_k, logits.size(-1))
        values, indices = torch.topk(logits, k, dim=-1)
        logits = values  # Only consider top-k
    
    # Apply temperature
    logits = logits / temperature
    
    # Sample from distribution
    probs = F.softmax(logits, dim=-1)
    if top_k is not None:
        sampled_idx = torch.multinomial(probs, num_samples=1)
        return indices.gather(-1, sampled_idx)  # Map back to vocab indices
    else:
        return torch.multinomial(probs, num_samples=1)

# Visualize temperature effect
logits = torch.tensor([[2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -2.5]])
temps = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, len(temps), figsize=(14, 3))
for ax, temp in zip(axes, temps):
    if temp == 0:
        probs = torch.zeros_like(logits)
        probs[0, 0] = 1.0
    else:
        probs = F.softmax(logits / temp, dim=-1)
    ax.bar(range(10), probs[0].numpy())
    ax.set_ylim(0, 1)
    ax.set_title(f'Temperature = {temp}')
    ax.set_xlabel('Token')
    if ax == axes[0]:
        ax.set_ylabel('Probability')

plt.tight_layout()
plt.show()

print("Temperature effects:")
print("  - T=0.1: Very deterministic (almost greedy)")
print("  - T=1.0: Standard sampling")
print("  - T=2.0: More random, explores alternatives")

## 7.5 Tool Use: Calculator

Nanochat supports tool use for computation. When the model outputs `<|python_start|>...<|python_end|>`, the expression is evaluated.

In [ ]:
def use_calculator(expr):
    """
    Safely evaluate a math expression.
    From nanochat/engine.py
    """
    # Remove commas from numbers
    expr = expr.replace(",", "")
    
    # Check if it's a safe math expression
    allowed = "0123456789*+-/.() "
    if not all(c in allowed for c in expr):
        return None
    
    # Disallow power operator (can be slow)
    if "**" in expr:
        return None
    
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return result
    except:
        return None

# Test the calculator
expressions = [
    "2 + 2",
    "123 * 456",
    "1000000 / 7",
    "(10 + 5) * 3",
    "1,000,000 + 500,000",  # Commas removed
]

print("Calculator examples:")
for expr in expressions:
    result = use_calculator(expr)
    print(f"  {expr} = {result}")

print("\nTool use flow:")
print("  1. Model outputs: <|python_start|>123 * 456<|python_end|>")
print("  2. Engine evaluates: 123 * 456 = 56088")
print("  3. Engine injects: <|output_start|>56088<|output_end|>")
print("  4. Model continues generation...")

## 7.6 Complete Engine Flow

The nanochat engine handles:
- KV cache management
- Tool use detection and execution
- Streaming generation

In [ ]:
class RowState:
    """Track state for each sample in a batch."""
    def __init__(self):
        self.tokens = []  # Generated tokens
        self.forced_tokens = []  # Tool output tokens to force
        self.in_python_block = False  # Are we inside <|python_start|>...<|python_end|>?
        self.python_tokens = []  # Tokens of current python expression
        self.completed = False  # Generation done?

def simulate_engine_flow():
    """Simulate the engine's generation flow."""
    
    # Simulated token sequence (in reality, would come from model)
    tokens = [
        "Let", " me", " calculate", ":", " ",
        "<|python_start|>", "25", " *", " 4", "<|python_end|>",
        # Engine inserts: <|output_start|>100<|output_end|>
        "The", " answer", " is", " ", "100", ".", "<|assistant_end|>"
    ]
    
    state = RowState()
    
    print("Engine generation flow:")
    print("="*60)
    
    output = []
    i = 0
    while i < len(tokens):
        token = tokens[i]
        
        if token == "<|python_start|>":
            state.in_python_block = True
            state.python_tokens = []
            output.append(token)
            print(f"  [{i}] {token} → Entering python block")
            
        elif token == "<|python_end|>" and state.in_python_block:
            state.in_python_block = False
            output.append(token)
            
            # Evaluate expression
            expr = "".join(state.python_tokens)
            result = use_calculator(expr)
            
            print(f"  [{i}] {token} → Exiting python block")
            print(f"       Evaluating: '{expr}' = {result}")
            
            # Inject result
            if result is not None:
                output.extend(["<|output_start|>", str(result), "<|output_end|>"])
                print(f"       Injected: <|output_start|>{result}<|output_end|>")
            
        elif state.in_python_block:
            state.python_tokens.append(token)
            output.append(token)
            print(f"  [{i}] {token} → Collecting for evaluation")
            
        elif token == "<|assistant_end|>":
            state.completed = True
            print(f"  [{i}] {token} → Generation complete")
            break
            
        else:
            output.append(token)
            print(f"  [{i}] {token}")
        
        i += 1
    
    print("\nFinal output:")
    print("".join(output))

simulate_engine_flow()

## 7.7 Performance Comparison

KV caching dramatically improves performance:

In [ ]:
# Theoretical performance comparison
def compute_flops(seq_len, hidden_dim, num_layers, with_cache=True):
    """
    Estimate FLOPs for generating `output_len` tokens.
    """
    # Attention FLOPs per token
    # Q @ K^T: seq_len * hidden_dim
    # Softmax: seq_len
    # Attn @ V: seq_len * hidden_dim
    
    if with_cache:
        # Only compute for new token attending to all cached
        return seq_len * hidden_dim * 2 * num_layers
    else:
        # Recompute full attention
        return seq_len * seq_len * hidden_dim * 2 * num_layers

# Compare for different sequence lengths
hidden_dim = 768
num_layers = 12
seq_lens = [64, 128, 256, 512, 1024, 2048]

print("FLOPs comparison (generating 1 token):")
print("="*60)
print(f"{'Seq Len':<10} {'With Cache':<15} {'Without Cache':<15} {'Speedup':<10}")
print("-"*60)

for seq_len in seq_lens:
    with_cache = compute_flops(seq_len, hidden_dim, num_layers, True)
    without_cache = compute_flops(seq_len, hidden_dim, num_layers, False)
    speedup = without_cache / with_cache
    print(f"{seq_len:<10} {with_cache/1e6:<15.1f}M {without_cache/1e6:<15.1f}M {speedup:<10.0f}x")

print("\nKV cache provides O(n) speedup per token!")

## 7.8 Connection to Mini-SGLang

For more advanced inference optimization, see the mini-sglang learning guide:

- **Paged Attention**: Memory-efficient KV cache with page tables
- **Continuous Batching**: Dynamic batching for higher throughput
- **Radix Cache**: Prefix caching for repeated prompts
- **Chunked Prefill**: Better memory/compute balance

## Summary

In this notebook, we learned:

1. ✅ **Autoregressive generation** - Token-by-token prediction
2. ✅ **KV Cache** - Store and reuse K, V tensors
3. ✅ **Prefill vs Decode** - Parallel input processing vs sequential generation
4. ✅ **Sampling** - Temperature and top-k
5. ✅ **Tool use** - Calculator integration
6. ✅ **Performance** - O(n) speedup with caching

## Next Steps

Continue to **[Module 8: Serving](08_serving.ipynb)** to learn:
- FastAPI server setup
- OpenAI-compatible API
- Multi-GPU worker pool

---

**Estimated time for this notebook: 45-60 minutes**